# Re-train from scratch using stored best hyperparameters

This notebook:

- Loads the DRIAMS-A pickle
- Loads best hyperparameters from saved models
- Re-trains models from scratch
- Uses 10 independent stratified 80/20 splits
- max_epochs = 1200
- early stopping patience = 50
- Computes metrics on each test split
- Returns averaged metrics

Metrics:
- Accuracy (per antibiotic, averaged)
- Hamming Loss (per antibiotic, averaged)
- Weighted F1 (per antibiotic, averaged)

Final summary table:
Species × {Single Label, LPS, Multilabel}

## Imports and Configuration

In [28]:
import os
import json
import pickle
import numpy as np
import pandas as pd
from tqdm import tqdm

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import f1_score, accuracy_score, hamming_loss

import torch
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

MAX_EPOCHS = 1200
PATIENCE = 50
N_SPLITS = 10
RANDOM_SEED = 42

## Define Models

In [29]:
class Identity(nn.Module):
    def forward(self, x):
        return x

def get_activation(name):
    if name == "relu":
        return nn.ReLU()
    if name == "tanh":
        return nn.Tanh()
    if name == "logistic":
        return nn.Sigmoid()
    if name == "identity":
        return Identity()

class MLP(nn.Module):
    def __init__(self, input_dim, output_dim, layer1, layer2, layer3, activation):
        super().__init__()
        act = get_activation(activation)
        self.net = nn.Sequential(
            nn.Linear(input_dim, layer1), act,
            nn.Linear(layer1, layer2), act,
            nn.Linear(layer2, layer3), act,
            nn.Linear(layer3, output_dim)
        )
    def forward(self, x):
        return self.net(x)

## Define metrics

In [30]:
def evaluate_multilabel(y_true, y_pred):
    n_labels = y_true.shape[1]
    wf1s, accs, hls = [], [], []
    for j in range(n_labels):
        wf1s.append(f1_score(y_true[:, j], y_pred[:, j], average="weighted"))
        accs.append(accuracy_score(y_true[:, j], y_pred[:, j]))
        hls.append(hamming_loss(y_true[:, j], y_pred[:, j]))
    return {
        "WF1": np.mean(wf1s),
        "ACC": np.mean(accs),
        "HL": np.mean(hls)
    }

## Define Early stopping

In [37]:
def train_model(model, optimizer, criterion,
                X_train, y_train,
                X_val, y_val,
                task_type):

    best_state = None
    best_score = -np.inf
    bad = 0

    Xtr = torch.tensor(X_train, dtype=torch.float32).to(device)
    Xva = torch.tensor(X_val, dtype=torch.float32).to(device)

    # ---------------- TARGET HANDLING ----------------
    if task_type == "binary":
        ytr = torch.tensor(y_train, dtype=torch.float32).view(-1,1).to(device)

    elif task_type == "multilabel":
        ytr = torch.tensor(y_train, dtype=torch.float32).to(device)

    elif task_type == "multiclass":  # LPS
        ytr = torch.tensor(y_train, dtype=torch.long).to(device)

    else:
        raise ValueError("Unknown task_type")

    loader = DataLoader(TensorDataset(Xtr, ytr), batch_size=128, shuffle=True)

    for epoch in range(MAX_EPOCHS):

        model.train()
        for xb, yb in loader:
            optimizer.zero_grad()
            logits = model(xb)
            loss = criterion(logits, yb)
            loss.backward()
            optimizer.step()

        model.eval()
        with torch.no_grad():
            logits_val = model(Xva)

        # ---------------- METRICS ----------------
        if task_type == "binary":
            preds = (torch.sigmoid(logits_val) > 0.5).cpu().numpy().ravel()
            score = f1_score(y_val, preds, average="weighted")

        elif task_type == "multilabel":
            preds = (torch.sigmoid(logits_val) > 0.5).cpu().numpy()
            score = evaluate_multilabel(y_val, preds)["WF1"]

        elif task_type == "multiclass":
            preds = torch.argmax(logits_val, dim=1).cpu().numpy()
            score = f1_score(y_val, preds, average="weighted")

        if score > best_score:
            best_score = score
            best_state = model.state_dict()
            bad = 0
        else:
            bad += 1
            if bad >= PATIENCE:
                break

    model.load_state_dict(best_state)
    return model

## Load data

In [32]:
PICKLE_PATH = "/export/usuarios01/egarroyo/MALDI_for_AMR_prediction/data/DRIAMS_A_AMR_paper_replication.pkl"
MODEL_ROOT = "/export/usuarios01/egarroyo/MALDI_for_AMR_prediction/saved_models"

with open(PICKLE_PATH, "rb") as f:
    payload = pickle.load(f)

X = np.asarray(payload["data"])
y_species = np.asarray(payload["label"])
amr = np.asarray(payload["amr"])
antibiotics = list(payload["antibiotics"])

amr_df = pd.DataFrame(amr, columns=antibiotics)
amr_df["species"] = y_species

## Define training pipeline

In [ ]:
def retrain_species(species):

    # -------- localizar última carpeta benchmark ----------
    sp_root = sorted([d for d in os.listdir(MODEL_ROOT) if d.startswith("benchmark")])[-1]
    sp_path = os.path.join(MODEL_ROOT, sp_root, species)

    summary = pd.read_csv(os.path.join(sp_path, f"{species}__summary.csv"))

    # antibióticos binarios entrenados
    ab_list = summary[summary.Task=="binary"]["Antibiotic"].tolist()

    # -------- filtrar datos exactamente como el script original ----------
    df_sp = amr_df[amr_df["species"] == species].copy()

    # drop NaNs solo en antibióticos relevantes
    df_sp = df_sp.dropna(subset=ab_list)

    # drop NaNs en features
    idx = df_sp.index.values
    feat_ok = ~np.isnan(X[idx]).any(axis=1)
    df_sp = df_sp.loc[idx[feat_ok]]

    if len(df_sp) == 0:
        print(f"{species}: no data after filtering — skipped")
        return None

    X_sp = X[df_sp.index]
    y_multi = df_sp[ab_list].astype(int).values

    results = {"Single":[], "LPS":[], "Multi":[]}

    for split in range(N_SPLITS):

        train_idx, test_idx = train_test_split(
            np.arange(len(X_sp)),
            test_size=0.2,
            stratify=y_multi,
            random_state=RANDOM_SEED + split
        )

        scaler = StandardScaler()
        X_train = scaler.fit_transform(X_sp[train_idx])
        X_test = scaler.transform(X_sp[test_idx])

        y_train_multi = y_multi[train_idx]
        y_test_multi = y_multi[test_idx]

        # ======================
        # Single Label (binarios independientes)
        # ======================
        preds_single = []

        for _, row in summary[summary.Task=="binary"].iterrows():

            hp = {k.replace("hp_",""):row[k] for k in row.index if k.startswith("hp_")}

            model = MLP(
                X_train.shape[1], 1,
                int(hp["layer1"]),
                int(hp["layer2"]),
                int(hp["layer3"]),
                hp["activation"]
            ).to(device)

            optimizer = torch.optim.Adam(model.parameters(), lr=float(hp["lr"]))
            criterion = nn.BCEWithLogitsLoss()

            y_train_bin = y_train_multi[:, ab_list.index(row["Antibiotic"])]

            model = train_model(
                model, optimizer, criterion,
                X_train, y_train_bin,
                X_train, y_train_bin,
                task_type="binary" 
            )

            with torch.no_grad():
                logits = model(torch.tensor(X_test, dtype=torch.float32).to(device))
                preds_single.append(
                    (torch.sigmoid(logits) > 0.5).cpu().numpy().ravel()
                )

        preds_single = np.column_stack(preds_single)
        results["Single"].append(evaluate_multilabel(y_test_multi, preds_single))

        # ======================
        # Direct Multilabel
        # ======================
        row = summary[summary.Task=="direct_multilabel"].iloc[0]
        hp = {k.replace("hp_",""):row[k] for k in row.index if k.startswith("hp_")}

        model = MLP(
            X_train.shape[1], len(ab_list),
            int(hp["layer1"]),
            int(hp["layer2"]),
            int(hp["layer3"]),
            hp["activation"]
        ).to(device)

        optimizer = torch.optim.Adam(model.parameters(), lr=float(hp["lr"]))
        criterion = nn.BCEWithLogitsLoss()

        model = train_model(
            model, optimizer, criterion,
            X_train, y_train_multi,
            X_train, y_train_multi,
            task_type="multilabel" 
        )

        with torch.no_grad():
            logits = model(torch.tensor(X_test, dtype=torch.float32).to(device))
            preds = (torch.sigmoid(logits) > 0.5).cpu().numpy()

        results["Multi"].append(evaluate_multilabel(y_test_multi, preds))

        # ======================
        # LPS
        # ======================
        row = summary[summary.Task=="lps_multiclass"].iloc[0]
        hp = {k.replace("hp_",""):row[k] for k in row.index if k.startswith("hp_")}

        patterns = ["".join(map(str, r)) for r in y_train_multi]
        classes = {p: i for i, p in enumerate(set(patterns))}
        y_train_class = np.array([classes[p] for p in patterns])

        model = MLP(
            X_train.shape[1], len(classes),
            int(hp["layer1"]),
            int(hp["layer2"]),
            int(hp["layer3"]),
            hp["activation"]
        ).to(device)

        optimizer = torch.optim.Adam(model.parameters(), lr=float(hp["lr"]))
        criterion = nn.CrossEntropyLoss()

        model = train_model(
            model, optimizer, criterion,
            X_train, y_train_class,
            X_train, y_train_class,
            task_type = "multiclass"
        )

        with torch.no_grad():
            logits = model(torch.tensor(X_test, dtype=torch.float32).to(device))
            pred_class = torch.argmax(logits, 1).cpu().numpy()

        inv = {v: k for k, v in classes.items()}
        preds = np.array([[int(c) for c in inv[i]] for i in pred_class])

        results["LPS"].append(evaluate_multilabel(y_test_multi, preds))

    final = {}
    for key in results:
        final[key] = pd.DataFrame(results[key]).mean().to_dict()

    return final

In [36]:
species_list = amr_df["species"].unique()

rows = []

for sp in tqdm(species_list):
    res = retrain_species(sp)
    rows.append({
        "Species": sp,

        "Single_ACC": res["Single"]["ACC"],
        "Single_HL": res["Single"]["HL"],
        "Single_WF1": res["Single"]["WF1"],

        "LPS_ACC": res["LPS"]["ACC"],
        "LPS_HL": res["LPS"]["HL"],
        "LPS_WF1": res["LPS"]["WF1"],

        "Multi_ACC": res["Multi"]["ACC"],
        "Multi_HL": res["Multi"]["HL"],
        "Multi_WF1": res["Multi"]["WF1"],
    })

summary_df = pd.DataFrame(rows)
summary_df

  0%|          | 0/4 [02:53<?, ?it/s]


RuntimeError: 0D or 1D target tensor expected, multi-target not supported